# Random Sample Imputation

Mean/median imputation shrinks variance by stacking every missing value at one fixed point. **Random sample imputation** instead fills each missing value with a value **randomly drawn from the column's own existing, non-missing values** — preserving the original distribution's shape and spread far more faithfully.

## 1. Setup and Data

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('train.csv',usecols=['Age','Fare','Survived'])

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
df.head()

In [ ]:
df.isnull().mean() * 100

In [ ]:
X = df.drop(columns=['Survived'])
y = df['Survived']

## 2. Numeric Column — `Age`

Start a new column as an exact copy of `Age` — only the missing entries will get overwritten.

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [ ]:
X_train

In [ ]:
X_train['Age_imputed'] = X_train['Age']
X_test['Age_imputed'] = X_test['Age']

### ⚠️ Important: Chained Assignment Is Broken on Current Pandas

`X_train['Age_imputed'][mask] = values` is chained indexing — indexing `X_train['Age_imputed']` first, *then* indexing into that result. As covered in the pandas-deprecation notebook earlier in this series, pandas 3.0's permanent Copy-on-Write means this either raises `ChainedAssignmentError` or silently does nothing. The fix: a single `.loc[]` call targeting rows and column together.

In [ ]:
X_train.loc[X_train['Age_imputed'].isnull(), 'Age_imputed'] = (
    X_train['Age'].dropna().sample(X_train['Age'].isnull().sum(), random_state=0).values
)
X_test.loc[X_test['Age_imputed'].isnull(), 'Age_imputed'] = (
    X_train['Age'].dropna().sample(X_test['Age'].isnull().sum(), random_state=0).values
)


Note `X_test`'s missing values are sampled from **`X_train`'s** `Age` column, not its own — correctly avoiding any leakage from the test set's own values.

In [ ]:
X_train['Age_imputed'][X_train['Age_imputed'].isnull()] = X_train['Age'].dropna().sample(X_train['Age'].isnull().sum()).values
X_test['Age_imputed'][X_test['Age_imputed'].isnull()] = X_train['Age'].dropna().sample(X_test['Age'].isnull().sum()).values

In [ ]:
X_train['Age'].dropna().sample(X_train['Age'].isnull().sum()).values

In [ ]:
X_train['Age'].isnull().sum()

### Visualizing the Effect

Unlike mean/median imputation's single spike, random sample imputation's "after" curve should closely **overlap** the original — that's the entire point of this technique.

*(`sns.distplot()` is removed on current seaborn — replaced with `sns.kdeplot()`, the direct equivalent of `hist=False`.)*

In [ ]:
sns.kdeplot(X_train['Age'], label='Original')
sns.kdeplot(X_train['Age_imputed'], label='Imputed')
plt.legend()
plt.show()


Confirming numerically: variance should be nearly **unchanged**, unlike mean/median imputation's shrinkage.

In [ ]:
print('Original variable variance: ', X_train['Age'].var())
print('Variance after random imputation: ', X_train['Age_imputed'].var())

In [ ]:
X_train[['Fare', 'Age', 'Age_imputed']].cov()

In [ ]:
X_train[['Age', 'Age_imputed']].boxplot()

### Making the Sample Reproducible Per Row — Seeding with Another Column

A plain random sample gives a *different* imputed value every time the cell re-runs — not reproducible. One pattern for deterministic-but-still-"random" imputation: seed each row's sample using another column's value from that same row, so the same input always produces the same imputed output. *(The original line referenced an undefined `observation` variable — completed below as a full, runnable function applied row-by-row.)*

In [ ]:
def seeded_sample_impute(row, source_col, seed_col, pool):
    if pd.isnull(row[source_col]):
        return pool.sample(1, random_state=int(row[seed_col])).values[0]
    return row[source_col]

age_pool = X_train['Age'].dropna()
X_train['Age_imputed_seeded'] = X_train.apply(
    lambda row: seeded_sample_impute(row, 'Age', 'Fare', age_pool), axis=1
)
X_train[['Age', 'Fare', 'Age_imputed_seeded']].head()


## 3. Categorical Columns — `GarageQual` and `FireplaceQu`

Exact same idea, applied to categorical columns: sample replacement values from the column's own existing categories, rather than always picking the single most frequent one (as mode imputation would).

In [ ]:
data = pd.read_csv('house-train.csv',usecols=['GarageQual','FireplaceQu', 'SalePrice'])

In [ ]:
data.head()

In [ ]:
data.isnull().mean() * 100

In [ ]:
X = data
y = data['SalePrice']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [ ]:
X_train['GarageQual_imputed'] = X_train['GarageQual']
X_test['GarageQual_imputed'] = X_test['GarageQual']

X_train['FireplaceQu_imputed'] = X_train['FireplaceQu']
X_test['FireplaceQu_imputed'] = X_test['FireplaceQu']

In [ ]:
X_train.sample(5)

### ⚠️ Same Chained-Assignment Fix Needed Here

Identical issue as `Age` above — fixed with `.loc[]`.

In [ ]:
X_train.loc[X_train['GarageQual_imputed'].isnull(), 'GarageQual_imputed'] = (
    X_train['GarageQual'].dropna().sample(X_train['GarageQual'].isnull().sum(), random_state=0).values
)
X_test.loc[X_test['GarageQual_imputed'].isnull(), 'GarageQual_imputed'] = (
    X_train['GarageQual'].dropna().sample(X_test['GarageQual'].isnull().sum(), random_state=0).values
)

X_train.loc[X_train['FireplaceQu_imputed'].isnull(), 'FireplaceQu_imputed'] = (
    X_train['FireplaceQu'].dropna().sample(X_train['FireplaceQu'].isnull().sum(), random_state=0).values
)
X_test.loc[X_test['FireplaceQu_imputed'].isnull(), 'FireplaceQu_imputed'] = (
    X_train['FireplaceQu'].dropna().sample(X_test['FireplaceQu'].isnull().sum(), random_state=0).values
)


Checking category proportions stayed close to the original — the key claim of this technique:

In [ ]:
temp = pd.concat(
        [
            X_train['GarageQual'].value_counts() / len(X_train['GarageQual'].dropna()),
            X_train['GarageQual_imputed'].value_counts() / len(X_train)
        ],
        axis=1)

temp.columns = ['original', 'imputed']

In [ ]:
temp

In [ ]:
temp = pd.concat(
        [
            X_train['FireplaceQu'].value_counts() / len(X_train['FireplaceQu'].dropna()),
            X_train['FireplaceQu_imputed'].value_counts() / len(df)
        ],
        axis=1)

temp.columns = ['original', 'imputed']

temp

Same `distplot` fix applies to the per-category `SalePrice` comparison below — replaced with `kdeplot`.

In [ ]:
for category in X_train['FireplaceQu'].dropna().unique():
    sns.kdeplot(X_train[X_train['FireplaceQu'] == category]['SalePrice'], label=category)
plt.legend()
plt.title('SalePrice by FireplaceQu (original)')
plt.show()


In [ ]:
for category in X_train['FireplaceQu_imputed'].dropna().unique():
    sns.kdeplot(X_train[X_train['FireplaceQu_imputed'] == category]['SalePrice'], label=category)
plt.legend()
plt.title('SalePrice by FireplaceQu (after random sample imputation)')
plt.show()


## Summary

| Task | Method |
|---|---|
| Fill missing values from the column's own distribution | `col.dropna().sample(n_missing).values` |
| Assign back safely | `.loc[mask, 'col'] = values` *(never chained `df['col'][mask] = ...`)* |
| Sample test's missing values from train only | always `.sample()` from `X_train`, never `X_test` |
| Check it preserved the distribution | `.var()`, overlaid `sns.kdeplot()` |

### Random sample vs. mean/median
Keeps variance and shape almost unchanged (the main advantage) — but introduces its own randomness, so results differ slightly each run unless a `random_state` is fixed, as done throughout above.